## Gold Layer - Vendor Performance

In [0]:
%run ./00_config

In [0]:
%sql
CREATE TABLE IF NOT EXISTS nyc_taxi_project.gold.vendor_performance
(
  pickup_year           INT,
  pickup_month          INT,
  vendor_name           STRING,
  pickup_borough        STRING,
  total_trips           LONG,
  total_revenue         DOUBLE,
  avg_fare              DOUBLE,
  avg_tip_pct           DOUBLE,
  avg_trip_distance     DOUBLE,
  avg_duration_minutes  DOUBLE,
  avg_passenger_count   DOUBLE,
  airport_trips         LONG,
  rush_hour_trips       LONG
)
USING DELTA
COMMENT 'Gold 3 - Vendor performance comparison (batch)';

In [0]:
from pyspark.sql import functions as F

def process_gold_vendor():

    df = spark.read.format("delta").table(SILVER2_TABLE)

    (df
        .withColumn("vendor_name",
            F.when(F.col("VendorID") == 1, "Creative Mobile Technologies")
             .when(F.col("VendorID") == 2, "VeriFone Inc")
             .otherwise("Unknown"))
        .groupBy(
            "pickup_year", "pickup_month",
            "vendor_name", "pickup_borough"
        )
        .agg(
            F.count("*")                              .alias("total_trips"),
            F.round(F.sum("total_amount"), 2)         .alias("total_revenue"),
            F.round(F.avg("fare_amount"), 2)          .alias("avg_fare"),
            F.round(F.avg("tip_percentage"), 2)       .alias("avg_tip_pct"),
            F.round(F.avg("trip_distance"), 2)        .alias("avg_trip_distance"),
            F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
            F.round(F.avg("passenger_count"), 2)      .alias("avg_passenger_count"),
            F.sum(F.col("is_airport_trip")
                   .cast("int"))                      .alias("airport_trips"),
            F.sum(F.col("is_rush_hour")
                   .cast("int"))                      .alias("rush_hour_trips")
        )
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(GOLD_VENDOR))

    count = spark.read.format("delta").table(GOLD_VENDOR).count()

process_gold_vendor()

In [0]:
%sql
SELECT
    vendor_name,
    pickup_year,
    SUM(total_trips)                AS total_trips,
    ROUND(SUM(total_revenue), 2)    AS total_revenue_usd,
    ROUND(AVG(avg_fare), 2)         AS avg_fare,
    ROUND(AVG(avg_tip_pct), 2)      AS avg_tip_pct,
    ROUND(AVG(avg_trip_distance),2) AS avg_distance_miles,
    SUM(airport_trips)              AS total_airport_trips,
    SUM(rush_hour_trips)            AS total_rush_hour_trips
FROM nyc_taxi_project.gold.vendor_performance
GROUP BY 1, 2
ORDER BY 2, total_revenue_usd DESC;